## THE CORRECT packages :/

In [ ]:
!pip install --upgrade pip
!pip install open3d
!pip install torch==2.2.2 torchvision==0.17.2 torchaudio==2.2.2 --index-url https://download.pytorch.org/whl/cu121
!pip install "numpy<2.0.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 22.0 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.7/447.7 MB 54.8 MB/s  0:00:05
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 51.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 69.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 49.2 MB/s  0:00:00
  Attempting uninstall: widgetsnbextension
    Found existing installation: widgetsnbextension 3.6.10
    Uninstalling widgetsnbextension-3.6.10:
      Successfully uninstalled widgetsnbextension-3.6.10
  Attempting uninstall: ipywidgets
    Found existing installation: ipywidgets 7.7.1
    Uninstalling ipywidgets-7.7.1:
      Successfully uninstalled ipywidgets-7.7.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10/10 [open3d]


## Environment Setup

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [20]:
import os

GOOGLE_DRIVE_PATH = "/content/drive/MyDrive/THESIS"
DATASET_PATH = "/content/drive/MyDrive/THESIS_other/mmw/rsu_1"
MYDATASET_LIDAR_PATH = "/content/drive/MyDrive/THESIS_other/mmw/MyDataset_rsu1/lidar"
MYDATASET_LIDAR_LABEL_PATH = "/content/drive/MyDrive/THESIS_other/mmw/MyDataset_rsu1/labels_lidar"

In [4]:
from posix import pipe
import open3d.ml as _ml3d
import open3d.ml.torch as ml3d
import torch
import open3d as o3d
import numpy as np

## Data Generation (LiDAR pcd & LiDAR labels)

## Load pretrained model

In [5]:
# pretrained model
framework = "torch"
device = "gpu" if torch.cuda.is_available() else "cpu"

# fetch config
# randlanet_semantickitti
cfg_file = os.path.join(GOOGLE_DRIVE_PATH, "open3d-ml/Open3D-ML/ml3d/configs/randlanet_semantickitti.yml")
cfg = _ml3d.utils.Config.load_from_file(cfg_file)

# model and pipeline
Model = _ml3d.utils.get_module("model", cfg.model.name, framework)
model = Model(**cfg.model)
pipeline = ml3d.pipelines.SemanticSegmentation(
    model,
    device=device,
    **cfg.pipeline
)

# weights
ckpt_folder = os.path.join(GOOGLE_DRIVE_PATH, "code", "logs")
os.makedirs(ckpt_folder, exist_ok=True)
ckpt_path = os.path.join(ckpt_folder, "randlanet_semantickitti_202201071330utc.pth")
randlanet_url = "https://storage.googleapis.com/open3d-releases/model-zoo/randlanet_semantickitti_202201071330utc.pth"
if not os.path.exists(ckpt_path):
    cmd = "wget {} -O {}".format(randlanet_url, ckpt_path)
    os.system(cmd)
pipeline.load_ckpt(ckpt_path=ckpt_path)

In [23]:
def LiDAR_DataLabel_Gen_single(sample_index):
  # load raw data preparation
  pcd_path = os.path.join(DATASET_PATH, f"{sample_index}.pcd")
  pcd = o3d.io.read_point_cloud(pcd_path)

  # save raw lidar pcd to dataset
  pcd_save_path = os.path.join(MYDATASET_LIDAR_PATH, f"{sample_index}.pcd")
  o3d.io.write_point_cloud(pcd_save_path, pcd)

  # form data to input to pretrained model
  points = np.asarray(pcd.points).astype(np.float32)
  xyz = np.asarray(pcd.points)
  data = {"point":torch.tensor(xyz),
          'feat': None,
          'label':torch.tensor(np.zeros((len(xyz),), dtype=np.int32))}

  # initialize labels array
  labels = -1 * np.ones(xyz.shape[0], dtype=int)    # -1:unlabeled, 0:cars, 1:buildings, 2:curbs & roads

  print(labels.shape)

  # static objects (buildings, poles, roads)
  # use pretrained model to segment
  result_segmodel = pipeline.run_inference(data)
  labels_segmodel = result_segmodel["predict_labels"]
  labels[(labels_segmodel == 17)] = 1  # building
  labels[(labels_segmodel == 12)] = 1  # building
  labels[(labels_segmodel == 16)] = 2  # curbs

  # moving objects (cars)
  # use position range to extract
  road1 = {
      'xpymin': 0,
      'xpymax': 15,
      'ymxmin': -60,
      'ymxmax': 10
  }
  road2 = {
      'ymxmin': -12,
      'ymxmax': -2.3,
      'xpymin': -10,
      'xpymax': 60
  }
  zmin, zmax = -3.8, 30

  for i in range(len(labels)):
      # road 1
      if road1['xpymin'] < xyz[i][0] + xyz[i][1] < road1['xpymax'] and road1['ymxmin'] < xyz[i][1] - xyz[i][0] < road1['ymxmax'] and zmin < xyz[i][2] < zmax:
          labels[i] = 0
      # road 2
      if road2['ymxmin'] < xyz[i][1] - xyz[i][0] < road2['ymxmax'] and road2['xpymin'] < xyz[i][0] + xyz[i][1] < road2['xpymax'] and zmin < xyz[i][2] < zmax:
          labels[i] = 0

  # save lidar point cloud labels
  labels_save_path = os.path.join(MYDATASET_LIDAR_LABEL_PATH, f"{sample_index}.npy")
  np.save(labels_save_path, labels)


In [24]:
index_range = ["016653", "016655"]
index = "016653"

while index != index_range[1]:
    print(index)
    LiDAR_DataLabel_Gen_single(index)
    index = f"{int(index) + 1:06d}"

016653
(28008,)



test 0/1: 100%|██████████| 27586/27586 [08:38<00:00, 53.19it/s]
/usr/local/lib/python3.12/dist-packages/open3d/_ml3d/torch/pipelines/semantic_segmentation.py:179: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(data['label']), model.cfg.num_classes,
/usr/local/lib/python3.12/dist-packages/open3d/_ml3d/torch/modules/metrics/semseg_metric.py:54: RuntimeWarning: Mean of empty slice
  accs.append(np.nanmean(accs))
/usr/local/lib/python3.12/dist-packages/open3d/_ml3d/torch/modules/metrics/semseg_metric.py:87: RuntimeWarning: Mean of empty slice
  ious.append(np.nanmean(ious))


016654
(28001,)


test 0/1: 100%|██████████| 27586/27586 [00:16<00:00, 1637.21it/s]
/usr/local/lib/python3.12/dist-packages/open3d/_ml3d/torch/pipelines/semantic_segmentation.py:179: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(data['label']), model.cfg.num_classes,
/usr/local/lib/python3.12/dist-packages/open3d/_ml3d/torch/modules/metrics/semseg_metric.py:54: RuntimeWarning: Mean of empty slice
  accs.append(np.nanmean(accs))
/usr/local/lib/python3.12/dist-packages/open3d/_ml3d/torch/modules/metrics/semseg_metric.py:87: RuntimeWarning: Mean of empty slice
  ious.append(np.nanmean(ious))
